# Conditional autoencoder architectures and training

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


**Defining the utils**

**COMMENT: restart after having run this cell, so it is in memory**

**Description**

1. Change the results path for the number hidden layer in each model, iteratively : results_CA_0, results_CA_1, results_CA_2, results_CA_0
2. Run the code that defines the desired model with the number of hidden layers
3. Run the codes that preprocess the data for the models.
4. Run the code for the cross-validation, which makes prediction on the validation set. The performance evaluation metrcs are saved in the results path.Remember to change the results path for the cross-validation.
5. Run the the code that predicts the desired model with the desired number of hidden layers. remember to change the results path for the out-of-sample  predictions on the test set. This code makes predictions for each latent latent factor across a time-series split, and the predicted results are saved.

**sliding window with validation**

In [ ]:
%%writefile utils.py
#!/usr/bin/env python
# -*- coding: utf-8 -*-
__author__ = 'Hans Andrias Gregoriussen'

import numpy as np

np.random.seed(42)


def format_time(t):
    """Return a formatted time string 'HH:MM:SS
    based on a numeric time() value"""
    m, s = divmod(t, 60)
    h, m = divmod(m, 60)
    return f'{h:0>2.0f}:{m:0>2.0f}:{s:0>2.0f}'

class MultipleTimeSeriesCV:
    """Generates tuples of train_idx, val_idx and test_idx pairs
    Assumes the MultiIndex contains levels 'symbol' and 'date'
    purges overlapping outcomes"""

    def __init__(self,
                 n_splits=3,
                 train_period_length=126,
                 test_period_length=21,
                 val_period_length=21,
                 lookahead=None,
                 date_idx='date',
                 shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle
        self.date_idx = date_idx





    def split(self, X, y=None, groups=None):
        unique_dates = X.index.get_level_values(self.date_idx).unique()
        days = sorted(unique_dates, reverse=True)
        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead
            split_idx.append([train_start_idx, train_end_idx,
                              val_start_idx, val_end_idx,
                              test_start_idx, test_end_idx])

        dates = X.reset_index()[[self.date_idx]]
        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:

            train_idx = dates[(dates[self.date_idx] > days[train_start])
                              & (dates[self.date_idx] <= days[train_end])].index
            val_idx = dates[(dates[self.date_idx] > days[val_start])
                              & (dates[self.date_idx] <= days[val_end])].index
            test_idx = dates[(dates[self.date_idx] > days[test_start])
                             & (dates[self.date_idx] <= days[test_end])].index
            if self.shuffle:
                np.random.shuffle(list(train_idx))
            yield train_idx.to_numpy(),val_idx.to_numpy(), test_idx.to_numpy()

    def get_n_splits(self, X, y, groups=None):
        return self.n_splits



class DateTimeSeriesCV:
    """Generates tuples of train_idx,val_idx and test_idx pairs
    Assumes the index is of datetime type
    purges overlapping outcomes"""

    def __init__(self, n_splits=3, train_period_length=126, val_period_length=21, test_period_length=21, lookahead=None, shuffle=False):
        self.n_splits = n_splits
        self.lookahead = lookahead
        if self.lookahead is None:
            self.lookahead = 0
        self.test_length = test_period_length
        self.val_length = val_period_length
        self.train_length = train_period_length
        self.shuffle = shuffle

    def split(self, X, y=None, groups=None):
        unique_dates = X.index.unique()
        days = sorted(unique_dates, reverse=True)

        split_idx = []
        for i in range(self.n_splits):
            test_end_idx = i * self.test_length
            test_start_idx = test_end_idx + self.test_length
            val_end_idx = test_start_idx + self.lookahead
            val_start_idx = val_end_idx + self.val_length + self.lookahead
            train_end_idx = val_start_idx + self.lookahead
            train_start_idx = train_end_idx + self.train_length + self.lookahead

            if train_start_idx >= len(days):
                break  # Stop if we've gone beyond the dataset

            split_idx.append([train_start_idx, train_end_idx, val_start_idx, val_end_idx, test_start_idx, test_end_idx])

        for train_start, train_end, val_start, val_end, test_start, test_end in split_idx:
            train_idx = X.index[(X.index > days[train_start]) & (X.index <= days[train_end])]
            val_idx = X.index[(X.index > days[val_start]) & (X.index <= days[val_end])]
            test_idx = X.index[(X.index > days[test_start]) & (X.index <= days[test_end])]

            if self.shuffle:
                train_idx = np.random.permutation(train_idx)

            yield train_idx, val_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, os
from time import time
from pathlib import Path
from itertools import product
from tqdm import tqdm

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
import time

import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dot, Reshape, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import TensorBoard
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import quantile_transform
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

from scipy.stats import spearmanr

from utils import format_time, MultipleTimeSeriesCV
from utils import format_time, DateTimeSeriesCV

from numpy.linalg import inv


**defining my results path**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/revised_results_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)


**load beta input data**

In [ ]:
from google.colab import drive
import pandas as pd
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/Tidy Finance ML dataset/beta_input_CRSP.parquet'

# Make sure merged_df1 is a DataFrame and not a function or any other type.
merged_df_padded = pd.read_parquet(file_path)


In [ ]:
print(merged_df_padded)
print(merged_df_padded.index)
print(merged_df_padded.shape)
print(type(merged_df_padded))


In [ ]:
import pandas as pd

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Function to find permnos with the same ret_excess value for each date
def find_same_value_permnos(group):
    ret_excess_values = group['ret_excess']
    common_value = ret_excess_values.mode().iloc[0]  # Get the most common value (mode)
    same_value_permnos = group[group['ret_excess'] == common_value]

    return common_value

# Apply the function to each group
common_values = grouped.apply(find_same_value_permnos)

# Display the results
print(common_values)


In [ ]:
import pandas as pd

# Assuming common_values is the series obtained from the previous step

# Function to replace common ret_excess values with zero
def replace_common_value_with_zero(df, date, common_value):
    df.loc[(df['date'] == date) & (df['ret_excess'] == common_value), 'ret_excess'] = 0

# Apply the function to each date and common value
for date, common_value in common_values.items():
    replace_common_value_with_zero(merged_df_padded, date, common_value)

# Display the updated DataFrame
print(merged_df_padded)


**Count common values for each date**

In [ ]:
import pandas as pd

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Function to find the common ret_excess value and count the permnos for each date
def find_common_value_and_count(group):
    ret_excess_values = group['ret_excess']
    common_value = ret_excess_values.mode().iloc[0]  # Get the most common value (mode)
    count = (ret_excess_values == common_value).sum()  # Count the permnos with the common value

    return pd.Series({'common_value': common_value, 'count': count})

# Apply the function to each group
common_values_and_counts = grouped.apply(find_common_value_and_count)

# Display the results
print(common_values_and_counts)
print(common_values_and_counts.mean())


In [ ]:
import pandas as pd

# Assuming merged_df_padded is the DataFrame we are working with
# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Function to find the common ret_excess value and count the permnos for each date
def find_common_value_and_count(group):
    ret_excess_values = group['ret_excess']
    common_value = ret_excess_values.mode().iloc[0]  # Get the most common value (mode)
    common_count = (ret_excess_values == common_value).sum()  # Count the permnos with the common value
    total_count = len(ret_excess_values)  # Total number of permnos for the date

    return pd.Series({'common_value': common_value, 'common_count': common_count, 'total_count': total_count})

# Apply the function to each group
common_values_and_counts = grouped.apply(find_common_value_and_count)

# Display the results
print(common_values_and_counts)


In [ ]:
# Function to calculate percentages
def calculate_percentages(df):
    df['percent_common'] = (df['common_count'] / df['total_count']) * 100
    df['percent_non_common'] = 100 - df['percent_common']
    return df

# Apply the function to calculate percentages
common_values_and_counts = calculate_percentages(common_values_and_counts)

# Display the updated DataFrame with percentages
print(common_values_and_counts)
print(common_values_and_counts.mean())


In [ ]:
print(grouped)

In [ ]:
import pandas as pd

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Step: Calculate the number of same values for each date in the 'ret_excess' column
same_value_counts_ret_excess = grouped.apply(lambda x: (x['ret_excess'] == x['ret_excess'].iloc[0]).sum())

# Display results
same_value_counts_ret_excess


In [ ]:
import pandas as pd

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Step: Check for the number of same values and capture the common value for each date in the 'ret_excess' column
results = []

for date, group in grouped:
    ret_excess_values = group['ret_excess']
    if ret_excess_values.nunique() == 1:
        common_value = ret_excess_values.iloc[0]
        results.append({'date': date, 'common_value': common_value})
    else:
        results.append({'date': date, 'common_value': None})

# Convert the results to a DataFrame
same_value_details_ret_excess = pd.DataFrame(results)

# Display the results
print(same_value_details_ret_excess)


In [ ]:
import pandas as pd

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Step 1: Calculate the average number of permnos with the same value for any column for each date
same_value_counts = grouped.apply(lambda x: (x.iloc[:, 2:] == x.iloc[:, 2:].iloc[0]).sum(axis=0))
average_same_value_counts = same_value_counts.mean()

# Step 2: Calculate the average number of permnos with the value zero for any column for each date
zero_value_counts = grouped.apply(lambda x: (x.iloc[:, 2:] == 0).sum(axis=0))
average_zero_value_counts = zero_value_counts.mean()

import ace_tools as tools; tools.display_dataframe_to_user("Average Same Value Counts", average_same_value_counts)
tools.display_dataframe_to_user("Average Zero Value Counts", average_zero_value_counts)


In [ ]:

# Group the DataFrame by date
grouped = merged_df_padded.groupby('date')

# Step 1: Calculate the same values for all permnos for any column for each date
same_value_details = grouped.apply(lambda x: x.iloc[:, 2:].apply(lambda col: (col == col.iloc[0]).all() * col.iloc[0]))

# Step 2: Calculate the number of permnos with the value zero for any column for each date
zero_value_counts = grouped.apply(lambda x: (x.iloc[:, 2:] == 0).sum(axis=0))
average_zero_value_counts = zero_value_counts.mean()

# Display results
same_value_details, average_zero_value_counts

print(same_value_details)
print(average_zero_value_counts)

In [ ]:
print("Average Number of Permnos with the Same Value for Any Column for Each Date:")
print(average_same_value_counts)


In [ ]:
print("Average Number of Permnos with the Value Zero for Any Column for Each Date:")
print(average_zero_value_counts)


Set frequency to monthly for the date level

In [ ]:
import pandas as pd

# Assuming merged_df_padded is your dataframe with columns 'date' and 'permno'

# Step 1: Set the 'date' and 'permno' columns as the MultiIndex
merged_df_padded.set_index(['date', 'permno'], inplace=True)

# Step 2: Extract the "date" level from the MultiIndex and convert to a DatetimeIndex
date_level = merged_df_padded.index.get_level_values('date')
date_level_with_freq = pd.DatetimeIndex(date_level).to_period('M').to_timestamp()

# Check the new date level
print("Date level with frequency set (sample):", date_level_with_freq[:10])
print("New date level frequency:", date_level_with_freq.freq)

# Step 3: Create a new MultiIndex with the updated date level
new_index = pd.MultiIndex.from_arrays(
    [date_level_with_freq, merged_df_padded.index.get_level_values('permno')],
    names=merged_df_padded.index.names
)

# Reindex the dataframe with this new MultiIndex
merged_df_padded.index = new_index

# Set the frequency explicitly on the 'date' level in the MultiIndex
merged_df_padded.index.levels[0].freq = 'MS'

# Validate the new index frequency
print("New index frequency:", merged_df_padded.index.levels[0].freq)  # Should print <MonthBegin>

# Check the unique dates to ensure they have the frequency set
print("Unique dates (sample):", merged_df_padded.index.levels[0][:10])

# Rename the column in the dataframe
merged_df_padded.rename(columns={'market_weighted_portfolio': 'characteristic_market_weighted_portfolio'}, inplace=True)

# Print the dataframe to confirm the change
print(merged_df_padded)


In [ ]:
print(merged_df_padded.index)

In [ ]:
# import pandas as pd

# # Assuming merged_df_padded is your dataframe with a MultiIndex ['date', 'permno']

# # 1. Extract the "date" level from the MultiIndex
# date_level = merged_df_padded.index.get_level_values('date')

# # 2. Convert to a DatetimeIndex and set the monthly frequency
# date_level_with_freq = pd.DatetimeIndex(date_level).to_period('M').to_timestamp()

# # Check the new date level
# print("Date level with frequency set (sample):", date_level_with_freq[:10])
# print("New date level frequency:", date_level_with_freq.freq)

# # 3. Create a new MultiIndex with the updated date level
# new_index = pd.MultiIndex.from_arrays(
#     [date_level_with_freq, merged_df_padded.index.get_level_values('permno')],
#     names=merged_df_padded.index.names
# )

# # Reindex the dataframe with this new MultiIndex
# merged_df_padded.index = new_index

# # Set the frequency explicitly on the 'date' level in the MultiIndex
# merged_df_padded.index.levels[0].freq = 'MS'

# # 4. Validate the new index frequency
# print("New index frequency:", merged_df_padded.index.levels[0].freq)  # Should print <MonthBegin>


# # Check the unique dates to ensure they have the frequency set
# print("Unique dates (sample):", merged_df_padded.index.levels[0][:10])


# # Rename the column in the dataframe
# merged_df_padded.rename(columns={'market_weighted_portfolio': 'characteristic_market_weighted_portfolio'}, inplace=True)

# # Print the dataframe to confirm the change
# print(merged_df_padded)


**load factor input data**

In [ ]:
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the file path
file_path = '/content/drive/My Drive/Tidy Finance ML dataset/x_t_all_months.parquet'

# Load the Parquet file into a DataFrame
portfolio_rets_df = pd.read_parquet(file_path)


# Display the first few rows of the DataFrame to verify it loaded correctly
print(portfolio_rets_df.head())
print(portfolio_rets_df.index)


set date to monthly frequency in the index

In [ ]:
import pandas as pd

# Assuming `portfolio_rets_df` is your dataframe with a "date" column

# 1. Convert the "date" column to datetime if it isn't already
portfolio_rets_df['date'] = pd.to_datetime(portfolio_rets_df['date'])

# 2. Set the "date" column as the index
portfolio_rets_df.set_index('date', inplace=True)

# 3. Ensure the index has the monthly frequency
# Convert the index to a DatetimeIndex and set the frequency to 'MS' (Month Start)
portfolio_rets_df.index = pd.DatetimeIndex(portfolio_rets_df.index).to_period('M').to_timestamp()

# Set the frequency explicitly on the index
portfolio_rets_df.index.freq = 'MS'

# 4. Validate the new index and display the dataframe
print("New index frequency:", portfolio_rets_df.index.freq)  # Should print <MonthBegin>
print(portfolio_rets_df.head())  # Display the first few rows to verify the change

# Optional: Check the unique dates to ensure they have the frequency set
print("Unique dates (sample):", portfolio_rets_df.index.unique()[:10])


**MODEL**

In [ ]:
x_t_all_months = portfolio_rets_df

# Define the list of characteristic codes
annual_codes = ["absacc", "acc", "age", "agr", "bm", "bm_ia", "cashdebt", "cashpr", "cfp", "cfp_ia",
                "chatoia", "chcsho", "chempia", "chinv", "chpmia", "convind", "currat", "depr", "divi",
                "divo", "dy", "egr", "ep", "gma", "grcapx", "grltnoa", "herf", "hire", "invest", "lev",
                "lgr", "mve_ia", "operprof", "orgcap", "pchcapx_ia", "pchcurrat", "pchdepr", "pchgm_pchsale",
                "pchquick", "pchsale_pchinvt", "pchsale_pchrect", "pchsale_pchxsga", "pchsaleinv", "pctacc",
                "ps", "quick", "rd", "rd_mve", "rd_sale", "realestate", "roic", "salecash", "saleinv", "salerec",
                "secured", "securedind", "sgr", "sin", "sp", "tang", "tb"]
quarterly_codes = ["aeavol", "cash", "chtx", "cinvest", "ear", "ms", "nincr", "roaq", "roavol", "roeq", "rsup",
                   "stdacc", "stdcf"]
monthly_codes = ["baspread", "beta", "betasq", "chmom", "dolvol", "idiovol", "ill", "indmom", "maxret", "mom12m",
                 "mom1m", "mom36m", "mom6m", "mvel1", "pricedelay", "retvol", "std_dolvol", "std_turn", "turn",
                 "zerotrade", "market_weighted_portfolio"]

# Combine all codes
all_codes = annual_codes + quarterly_codes + monthly_codes
narratives = [f'characteristic_{code}' for code in all_codes]
print(narratives)
print(merged_df_padded.index)

x_t_all_months = x_t_all_months[narratives]

n_factors = 6
n_narratives = len(narratives)
n_tickers = len(merged_df_padded.index.unique('permno'))

print(n_narratives)
print(n_tickers)

gpu_devices = tf.config.experimental.list_physical_devices('GPU')
if gpu_devices:
    print('Using GPU')
    tf.config.experimental.set_memory_growth(gpu_devices[0], True)
else:
    print('Using CPU')

**without regularization**

In [ ]:
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model

def make_model(hidden_units=32, n_factors=n_factors, n_tickers=n_tickers, n_narratives=n_narratives, return_components=False):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    hidden_layer = Dense(units=hidden_units, activation='relu', name='hidden_layer')(input_beta)
    batch_norm = BatchNormalization(name='batch_norm')(hidden_layer)

    output_beta = Dense(units=n_factors, name='output_beta')(batch_norm)
    output_factor = Dense(units=n_factors, name='output_factor')(input_factor)

    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output, name='FactorInteractionModel')
    model.compile(loss='mse', optimizer='adam')

    if return_components:
        # Create a standalone factor network model
        factor_model = Model(inputs=input_factor, outputs=output_factor, name='FactorNetwork')
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default


**special lasso on loss function**

**special loss function with factor prediction**

**with regularization**

**CA_0**

In [ ]:
from tensorflow.keras.layers import Input, Dense, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # Directly connect input_beta to output_beta with no hidden layer and no batch normalization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(input_beta)

    # Apply L1 regularization to the output_factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    output = Dot(axes=(2,1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

main_model.summary()
factor_network_model.summary()


**CA_1**

In [ ]:
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    hidden_layer = Dense(units=32, activation='relu', name='hidden_layer', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm = BatchNormalization(name='batch_norm')(hidden_layer)

    # Apply L1 regularization to the output_beta layer
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm)

    # Apply L1 regularization to the output_factor layer
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    output = Dot(axes=(2,1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

main_model.summary()
factor_network_model.summary()

**ca_2**

In [ ]:
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default


# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

# Print the summaries of the models
main_model.summary()
factor_network_model.summary()


**CA_3**

In [ ]:
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1

def make_model(n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((n_tickers, n_narratives), name='input_beta')
    input_factor = Input((n_narratives,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(batch_norm_2)
    batch_norm_3 = BatchNormalization(name='batch_norm_3')(hidden_layer_2)


    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_3)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default


# To get both the main model and the factor network model
main_model, factor_network_model = make_model(n_factors=6, return_components=True, l1_reg=0.001)

# To get only the main model
main_model = make_model(n_factors=6, return_components=False, l1_reg=0.01)

# Print the summaries of the models
main_model.summary()
factor_network_model.summary()


**Cross Validation**

**input preprocessing function**

In [ ]:
def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
    # Process merged_df_padded data
    train_merged, val_merged, test_merged = merged_data.iloc[train_idx_merged], merged_data.iloc[val_idx_merged], merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)
    X1_val = val_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)
    X1_test = test_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)
    # print(train_merged.index.get_level_values('date').unique())
    # print(val_merged.index.get_level_values('date').unique())
    # print(test_merged.index.get_level_values('date').unique())

    y_train = train_merged.ret_excess.unstack('permno')
    y_val = val_merged.ret_excess.unstack('permno')
    y_test = test_merged.ret_excess.unstack('permno')

    # Process loaded_portfolio_rets data
    X2_train = portfolio_data.loc[train_dates_portfolio]
    X2_val = portfolio_data.loc[val_dates_portfolio]
    X2_test = portfolio_data.loc[test_dates_portfolio]

    # Process loaded_portfolio_rets data
    # print("Train dates portfolio:", train_dates_portfolio)
    # print("Validation dates portfolio:", val_dates_portfolio)
    # print("Test dates portfolio:", test_dates_portfolio)

#    print("Portfolio data index:", portfolio_data.index)

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test


**CROSS VALIDATION**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/revised_results_CA_2testie')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

import time
from tensorflow.keras.callbacks import EarlyStopping
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr

early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['l1_reg', 'units', 'n_factors', 'fold', 'ic_mean', 'ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']

max_epochs = 100
factor_opts = [1, 2, 3, 4, 5, 6]
l1_reg = [0.000000001, 0.00000001, 0.0000001,0.000001, 0.00001, 0.0001,0.001, 0.01, 0.1]
#
param_grid = list(product(l1_reg, factor_opts))
units = 32
batch_size = 32

YEAR = 1
cv_merged = MultipleTimeSeriesCV(n_splits=15, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=20*YEAR, val_period_length=2*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Create the HDF5 file initially in 'table' format
with pd.HDFStore(str(results_path / 'scores.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
    store.close()

for l1_reg, n_factors in param_grid:
    scores = []
    factor_scores = []
    main_model, factor_network_model = make_model(n_factors=n_factors, return_components=True, l1_reg=l1_reg)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Training the whole model
        main_model.fit([X1_train, X2_train], y_train,
                       batch_size=batch_size,
                       validation_data=([X1_val, X2_val], y_val),
                       epochs=max_epochs,
                       verbose=0,  # Adjust verbosity as needed
                       callbacks=[early_stopping])  # Include both callbacks

        result = (pd.DataFrame({'y_pred': main_model.predict([X1_val, X2_val]).reshape(-1),
                                'y_true': y_val.stack().values},
                               index=y_val.stack().index)
                  .replace(-2, np.nan).dropna())

        # Calculate spearman's coefficient for val set
        r0 = spearmanr(result.y_true, result.y_pred)[0]

        # Calculate the daily spearman's coefficient
        r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])

        # Calculate R^2
        r2 = r2_score(result.y_true, result.y_pred)

        # Calculate MSE
        mse = mean_squared_error(result.y_true, result.y_pred)

        # Compile the factor network model
        factor_network_model.compile(optimizer='adam', loss='mse')

        # Factors for training set
        factor_outputs_train = factor_network_model.predict(X2_train)
        factor_df_train = pd.DataFrame(factor_outputs_train, index=y_train.stack().index.get_level_values('date').unique(), columns=[f'Factor_{i}' for i in range(factor_outputs_train.shape[1])])

        # Factors from validation set
        factor_outputs_val = factor_network_model.predict(X2_val)
        factor_df_val = pd.DataFrame(factor_outputs_val, index=y_val.stack().index.get_level_values('date').unique(), columns=[f'Factor_{i}' for i in range(factor_outputs_val.shape[1])])
        factor_std = np.std(factor_outputs_val, axis=0)

        print(f'{time.time()-start:.2f} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | sp-coe: {r0:6.2%} | '
              f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%}| R^2:{r2:6.2%}| MSE {mse:6.2%}')

        scores.append([l1_reg, units, n_factors, fold, r0, r1.mean(), r1.std(), r1.median(), r2, mse])

    scores_df = pd.DataFrame(scores, columns=cols)
    scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{l1_reg}/{n_factors}', append=True, format='table')

print("Training complete.")


In [ ]:
print(X1_train)
print(X2_train)
# print(X1_train.index)
# print(X1_train.shape)
# print(X2_train)
# print(X2_train.index)
# print(X2_train.shape)
# print(y_train)
# print(y_train.index)
# print(y_train.shape)


**without facotr SR max**


In [ ]:
# import time
# from tensorflow.keras.callbacks import Callback
# from scipy.optimize import minimize
# import numpy as np
# early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# cols = ['l1_reg','units', 'n_factors', 'fold', 'ic_mean','ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse', 'MVE SR', 'SRMAX', 'r-SRMAX', 'sd-SRMAX']



# max_epochs = 100
# n_factors = [1, 2, 3, 4, 5, 6]

# l1_reg = [0.000000001,0.000000005,0.00000001,0.00000005,0.0000001, 0.0000005, 0.000001]

# param_grid = list(product(l1_reg, n_factors))
# units=32
# batch_size = 32

# YEAR = 12
# cv_merged = MultipleTimeSeriesCV(n_splits=8, train_period_length=5*YEAR,val_period_length=3*YEAR, test_period_length=1*YEAR, lookahead=1*YEAR)
# cv_portfolio = DateTimeSeriesCV(n_splits=8, train_period_length=5*YEAR,val_period_length=3*YEAR , test_period_length=1*YEAR, lookahead=1*YEAR)


# start = time.time()

# for l1_reg, n_factors in param_grid:
#     scores = []
#     factor_scores = []
#     main_model, factor_network_model = make_model(hidden_units=32, n_factors=n_factors, return_components=True, l1_reg=l1_reg)

#     for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(loaded_portfolio_rets))):
#         X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, loaded_portfolio_rets, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)
#         ##Training the whole model
#         # In your training loop
#         main_model.fit([X1_train, X2_train], y_train,
#                   batch_size=batch_size,
#                   validation_data=([X1_val, X2_val], y_val),
#                   epochs=max_epochs,
#                   verbose=1,  # Adjust verbosity as needed
#                   callbacks=[early_stopping])  # Include both callbacks

#         result = (pd.DataFrame({'y_pred': main_model.predict([X1_val,
#                                                         X2_val]).reshape(-1),
#                                 'y_true': y_val.stack().values},
#                               index=y_val.stack().index)
#                   .replace(-2, np.nan).dropna())
#         #Calculate spearmans coefficient for val set
#         r0 = spearmanr(result.y_true, result.y_pred)[0]
#         # Calculate the daily spearman's coefficient
#         r1 = result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred,x.y_true)[0])
#         # calculate R^2
#         r2 = r2_score(result.y_true, result.y_pred)
#        # Calculate MSE
#         mse = mean_squared_error(result.y_true, result.y_pred)

#         # Assuming the rest of your code remains the same and just after the factor_network_model is created
#         factor_network_model.compile(optimizer='adam', loss='mse')

#         # Assuming `dates` is a list or array of date strings or objects corresponding to each row in `factor_outputs_val`
#         # FACTORS FOR TRAINING SET
#         factor_outputs_train = factor_network_model.predict(X2_train)
#         factor_df_train = pd.DataFrame(factor_outputs_train, index=y_train.stack().index.get_level_values('date').unique(), columns=[f'Factor_{i}' for i in range(factor_outputs_train.shape[1])])


#         # FACTORS FROM VALIDATION SET
#         factor_outputs_val = factor_network_model.predict(X2_val)

#         factor_df_val = pd.DataFrame(factor_outputs_val, index=y_val.stack().index.get_level_values('date').unique(), columns=[f'Factor_{i}' for i in range(factor_outputs_val.shape[1])])
#         factor_std = np.std(factor_outputs_val, axis=0)

#         # MEAN OF FACTORS FROM TRAINING SET
#         mean_train_factors = factor_df_val.mean()
#         cov_train_factors = factor_df_val.cov()


#         # MEAN VARIANCE PORTFOLIO OPTIMIZATION USIN MEAN AND COVARIANCE OF FACTORS FROM TRAIN SET TO CALCULATE OPTIMIZED AND FACTOR RETURNS FROM VAL SET.
#         initial_guess = np.ones(n_factors) / n_factors  # Equal allocation to each factor
#         optimized_result = max_sharpe_ratio(factor_df_val, initial_guess, n_factors, short=True)  # Set short=True to allow short selling
#         optimized_weights = optimized_result.x


#         r, sd = portfolio_performance(optimized_weights, factor_df_val)

#         sharpe_ratio = r / sd

#         print(f'{format_time(time.time()-start)} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | sp-coe: {r0:6.2%} | '
#               f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%}| R^2:{r2:6.2%}| MSE {mse:6.2%}|factors std {factor_std} | MAXSR: {sharpe_ratio}|MVE r:{r} MVE sd: {sd}' )
#         scores.append([l1_reg ,units, n_factors, fold, r0,
#                           r1.mean(), r1.std(), r1.median(), r2, mse, factor_std, sharpe_ratio, r, sd])

#         scores = pd.DataFrame(scores, columns=cols)
#         scores.to_hdf(results_path / 'scores.h5', f'{l1_reg}/{n_factors}')




**TROUBLESHOOT*

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 2: Import necessary module
from pathlib import Path

# Step 3: Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/results_CA_3')

# Step 4: Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Step 5: Check if the results path exists, if not, create it
if not results_path.exists():
    # Create the directory and any intermediate directories if necessary
    results_path.mkdir(parents=True)

# Optional: Print the path to confirm
print(f"Results path is set to: {results_path}")


In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

scores = []
with pd.HDFStore(results_path / 'scores.h5') as store:
    for key in store.keys():
        scores.append(store[key])
scores = pd.concat(scores)

In [ ]:
scores.info()

In [ ]:
# Correct way to group by 'n_factors' and 'l1_reg' and calculate mean for specified columns
avg = (scores.groupby(['n_factors', 'l1_reg'])
       [['ic_mean', 'ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']]
       .mean().reset_index())

# Display the resulting DataFrame
print(avg)


In [ ]:
avg.nlargest(n=20, columns=['r2_score'])

In [ ]:
# Sort by 'n_factors' and 'r2_score' in descending order
sorted_avg = avg.sort_values(by=['n_factors', 'r2_score'], ascending=[True, False])

# Drop duplicates, keeping the first entry of each 'n_factors' group
highest_r2_per_factor = sorted_avg.drop_duplicates(subset=['n_factors'], keep='first')

# Display the resulting DataFrame
print(highest_r2_per_factor)


**Single Model predictions on test set**

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
import time
from pathlib import Path

# Setup the training configuration
max_epochs = 100
batch_size = 32
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# Extract unique combinations of l1_reg and n_factors from highest_r2_per_factor
selected_combinations = highest_r2_per_factor[['l1_reg', 'n_factors']].drop_duplicates().to_records(index=False)

# Define cross-validation configurations
YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Define a new results path for the HDF5 file and predictions
new_results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3')
new_hdf5_path = new_results_path / 'new_scores.h5'

# Ensure the new results directory exists
if not new_results_path.exists():
    new_results_path.mkdir(parents=True)

# Create the HDF5 file initially in 'table' format if it doesn't exist
if not new_hdf5_path.exists():
    with pd.HDFStore(new_hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.close()

# Dictionary to accumulate predictions by n_factors
accumulated_predictions_by_n_factors = {}

# Iterate over the selected combinations only
for l1_reg, n_factors in selected_combinations:
    scores = []
    combined_predictions = []  # To accumulate predictions across folds

    # Create models based on the given parameters
    main_model, factor_network_model = make_model(n_factors=n_factors, return_components=True, l1_reg=l1_reg)

    # Cross-validation and training loop
    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Train the main model
        main_model.fit([X1_train, X2_train], y_train,
                       batch_size=batch_size,
                       validation_data=([X1_val, X2_val], y_val),
                       epochs=max_epochs,
                       verbose=1,  # Adjust verbosity as needed
                       callbacks=[early_stopping])

        # Test set predictions and performance metrics
        test_predictions = main_model.predict([X1_test, X2_test]).reshape(-1)
        test_result = pd.DataFrame({'y_pred': test_predictions, 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        test_spearman = spearmanr(test_result.y_true, test_result.y_pred)[0]
        daily_test_spearman = test_result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])
        test_r2 = r2_score(test_result.y_true, test_result.y_pred)
        test_mse = mean_squared_error(test_result.y_true, test_result.y_pred)

        # Store test set performance
        print(f'{time.time()-start:.2f} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | '
              f'Test SP-coef: {test_spearman:6.2%} | Test R^2:{test_r2:6.2%} | Test MSE {test_mse:6.2%}')

        scores.append([l1_reg, n_factors, fold, 'N/A', 'N/A', 'N/A', 'N/A', test_r2, test_mse, 'Test'])

        prediction_df = pd.DataFrame({
            'date': test_result.index.get_level_values('date'),
            'permno': test_result.index.get_level_values('permno').astype(str),
            'predicted returns': test_result['y_pred'],
            'actual predictions': test_result['y_true']
        }).reset_index(drop=True)  # Reset index to avoid multi-index issues

        # Append current fold's predictions to combined_predictions
        combined_predictions.append(prediction_df)

    # Concatenate predictions from all folds for the current l1_reg and n_factors
    if combined_predictions:
        all_predictions_df = pd.concat(combined_predictions)

        # Check if n_factors key already exists in the dictionary
        if n_factors not in accumulated_predictions_by_n_factors:
            accumulated_predictions_by_n_factors[n_factors] = []

        # Append the combined DataFrame for the current l1_reg to the list for this n_factors
        accumulated_predictions_by_n_factors[n_factors].append(all_predictions_df)

    # Convert scores to a DataFrame and append to the HDF5 file
    scores_df = pd.DataFrame(scores, columns=['l1_reg', 'n_factors', 'fold', 'spearman', 'daily_spearman_mean', 'daily_spearman_std', 'daily_spearman_median', 'r2_score', 'mse', 'set'])
    with pd.HDFStore(new_hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        scores_df.to_hdf(store, key=f'scores/{l1_reg}/{n_factors}', format='table')

# After processing all combinations, save the accumulated predictions for each n_factors
for n_factors, df_list in accumulated_predictions_by_n_factors.items():
    # Concatenate all DataFrames for this n_factors
    final_combined_df = pd.concat(df_list).reset_index(drop=True)

    # Save the final combined DataFrame for each n_factors in the HDF5 file
    n_factors_key = f'predictions/{n_factors}'
    with pd.HDFStore(new_hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
        store.put(n_factors_key, final_combined_df, format='table')
        print(f"Saved combined predictions for n_factors: {n_factors} under key: {n_factors_key}")

print("All predictions have been combined and saved.")


**Loading predictions that have been saved**

In [ ]:
import pandas as pd
from pathlib import Path

# Define the HDF5 file path
hdf5_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5')

# Open the HDF5 file to list all keys and load predictions
with pd.HDFStore(hdf5_path, mode='r') as store:
    keys = store.keys()
    print("Available keys in the HDF5 file:")
    for key in keys:
        print(key)

    # Dictionary to store loaded predictions grouped by n_factors
    predictions_by_n_factors = {}

    # Filter keys for combined predictions by n_factors
    for key in keys:
        if key.startswith('/predictions/') and key.count('/') == 2:  # This ensures we're looking for keys like 'predictions/n_factors'
            n_factors = key.split('/')[-1]  # Extract the n_factors value from the key

            # Load the predictions DataFrame
            predictions_df = store.get(key)

            # Add the predictions DataFrame to the dictionary for this n_factors
            predictions_by_n_factors[n_factors] = predictions_df

            print(f"Loaded combined predictions for n_factors: {n_factors}, key: {key}")
            print(predictions_df.head())  # Display the first few rows

# Now predictions_by_n_factors is a dictionary where keys are n_factors and values are the combined DataFrames


**Controlling for date range for saved predictions**

In [ ]:
import pandas as pd
from pathlib import Path

# Define the HDF5 file path
hdf5_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5')

# Open the HDF5 file to list all keys and load predictions
with pd.HDFStore(hdf5_path, mode='r') as store:
    keys = store.keys()
    print("Available keys in the HDF5 file:")
    for key in keys:
        print(key)

    # Dictionary to store loaded predictions grouped by n_factors
    predictions_by_n_factors = {}

    # Filter keys for combined predictions by n_factors
    for key in keys:
        if key.startswith('/predictions/') and key.count('/') == 2:  # This ensures we're looking for keys like 'predictions/n_factors'
            n_factors = key.split('/')[-1]  # Extract the n_factors value from the key

            # Load the predictions DataFrame
            predictions_df = store.get(key)

            # Add the predictions DataFrame to the dictionary for this n_factors
            predictions_by_n_factors[n_factors] = predictions_df

            print(f"Loaded combined predictions for n_factors: {n_factors}, key: {key}")
            print(predictions_df.head())  # Display the first few rows

# Calculate and print the date range for each n_factors DataFrame
for n_factors, df in predictions_by_n_factors.items():
    # Ensure the 'date' column is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Calculate the date range
    date_range = (df['date'].min(), df['date'].max())

    # Print the date range for the current n_factors
    print(f"Date range for n_factors {n_factors}: {date_range[0]} to {date_range[1]}")


**calculating average number of permno's per date**

In [ ]:
import pandas as pd
from pathlib import Path

# Define the HDF5 file path
hdf5_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results_CA_3/new_scores.h5')

# Open the HDF5 file to list all keys and load predictions
with pd.HDFStore(hdf5_path, mode='r') as store:
    keys = store.keys()
    print("Available keys in the HDF5 file:")
    for key in keys:
        print(key)

    # Dictionary to store loaded predictions grouped by n_factors
    predictions_by_n_factors = {}

    # Filter keys for combined predictions by n_factors
    for key in keys:
        if key.startswith('/predictions/') and key.count('/') == 2:  # This ensures we're looking for keys like 'predictions/n_factors'
            n_factors = key.split('/')[-1]  # Extract the n_factors value from the key

            # Load the predictions DataFrame
            predictions_df = store.get(key)

            # Add the predictions DataFrame to the dictionary for this n_factors
            predictions_by_n_factors[n_factors] = predictions_df

            print(f"Loaded combined predictions for n_factors: {n_factors}, key: {key}")
            print(predictions_df.head())  # Display the first few rows

# Calculate and print the date range and average permnos per date for each n_factors DataFrame
for n_factors, df in predictions_by_n_factors.items():
    # Ensure the 'date' column is in datetime format
    df['date'] = pd.to_datetime(df['date'])

    # Calculate the date range
    date_range = (df['date'].min(), df['date'].max())
    print(f"Date range for n_factors {n_factors}: {date_range[0]} to {date_range[1]}")

    # Calculate the number of unique permnos per date
    unique_permnos_per_date = df.groupby('date')['permno'].nunique()

    # Calculate the average number of unique permnos per date
    average_permnos_per_date = unique_permnos_per_date.mean()
    print(f"Average number of unique permnos per date for n_factors {n_factors}: {average_permnos_per_date:.2f}")



In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mean_squared_error, r2_score
# from scipy.stats import spearmanr
# import time
# from pathlib import Path

# # Setup the training configuration
# max_epochs = 100
# units = 32
# batch_size = 32
# early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# # Extract unique combinations of l1_reg and n_factors from highest_r2_per_factor
# selected_combinations = highest_r2_per_factor[['l1_reg', 'n_factors']].drop_duplicates().to_records(index=False)

# # Define cross-validation configurations
# YEAR = 12
# cv_merged = MultipleTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
# cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# start = time.time()

# # Define a new results path for the HDF5 file
# new_results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results')
# new_hdf5_path = new_results_path / 'new_scores.h5'

# # Ensure the new results directory exists
# if not new_results_path.exists():
#     new_results_path.mkdir(parents=True)

# # Create the HDF5 file initially in 'table' format if it doesn't exist
# if not new_hdf5_path.exists():
#     with pd.HDFStore(new_hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
#         store.close()

# # Iterate over the selected combinations only
# for l1_reg, n_factors in selected_combinations:
#     scores = []
#     # Create models based on the given parameters
#     main_model, factor_network_model = make_model(hidden_units=units, n_factors=n_factors, return_components=True, l1_reg=l1_reg)

#     # Cross-validation and training loop
#     for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
#         X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

#         # Train the main model
#         main_model.fit([X1_train, X2_train], y_train,
#                        batch_size=batch_size,
#                        validation_data=([X1_val, X2_val], y_val),
#                        epochs=max_epochs,
#                        verbose=1,  # Adjust verbosity as needed
#                        callbacks=[early_stopping])

#         # Test set predictions and performance metrics
#         test_predictions = main_model.predict([X1_test, X2_test]).reshape(-1)
#         test_result = pd.DataFrame({'y_pred': test_predictions, 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

#         test_spearman = spearmanr(test_result.y_true, test_result.y_pred)[0]
#         daily_test_spearman = test_result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])
#         test_r2 = r2_score(test_result.y_true, test_result.y_pred)
#         test_mse = mean_squared_error(test_result.y_true, test_result.y_pred)

#         # Store test set performance
#         print(f'{time.time()-start:.2f} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | '
#               f'Test SP-coef: {test_spearman:6.2%} | Test R^2:{test_r2:6.2%} | Test MSE {test_mse:6.2%}')

#         scores.append([l1_reg, units, n_factors, fold, 'N/A', 'N/A', 'N/A', 'N/A', test_r2, test_mse, 'Test'])

#     # Convert scores to a DataFrame and append to the new HDF5 file
#     scores_df = pd.DataFrame(scores, columns=['l1_reg', 'units', 'n_factors', 'fold', 'spearman', 'daily_spearman_mean', 'daily_spearman_std', 'daily_spearman_median', 'r2_score', 'mse', 'set'])
#     scores_df.to_hdf(new_hdf5_path, key=f'{l1_reg}/{n_factors}', append=True, format='table')

# print("Training and test predictions complete.")


**Ensemble**

In [ ]:
# from tensorflow.keras.callbacks import EarlyStopping
# import numpy as np
# import pandas as pd
# from sklearn.metrics import mean_squared_error, r2_score
# from scipy.stats import spearmanr
# import time
# from pathlib import Path
# import tensorflow as tf
# from tqdm import tqdm  # Progress bar

# # Setup the training configuration
# max_epochs = 100
# units = 32
# batch_size = 32
# early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# # Define the ensemble size
# ensemble_size = 10

# # Extract unique combinations of l1_reg and n_factors from highest_r2_per_factor
# selected_combinations = highest_r2_per_factor[['l1_reg', 'n_factors']].drop_duplicates().to_records(index=False)

# # Define cross-validation configurations
# YEAR = 12
# cv_merged = MultipleTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
# cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# start = time.time()

# # Define a new results path for the HDF5 file
# new_results_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/new_results')
# new_hdf5_path = new_results_path / 'new_scores.h5'

# # Ensure the new results directory exists
# if not new_results_path.exists():
#     new_results_path.mkdir(parents=True)

# # Create the HDF5 file initially in 'table' format if it doesn't exist
# if not new_hdf5_path.exists():
#     with pd.HDFStore(new_hdf5_path, mode='a', complevel=9, complib='blosc:zstd') as store:
#         store.close()

# # Iterate over the selected combinations only
# for l1_reg, n_factors in tqdm(selected_combinations):
#     scores = []

#     # Cross-validation and training loop
#     for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
#         X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

#         # Initialize a list to store predictions from each model in the ensemble
#         ensemble_predictions = []

#         # Use tqdm to add a progress bar for the ensemble loop
#         for seed in tqdm(range(ensemble_size), desc=f"Processing ensemble for l1_reg={l1_reg}, n_factors={n_factors}, fold={fold}"):
#             # Set the seed for reproducibility
#             tf.keras.utils.set_random_seed(seed)

#             # Create models based on the given parameters
#             main_model, _ = make_model(hidden_units=units, n_factors=n_factors, return_components=True, l1_reg=l1_reg)

#             # Train the main model
#             main_model.fit([X1_train, X2_train], y_train,
#                            batch_size=batch_size,
#                            validation_data=([X1_val, X2_val], y_val),
#                            epochs=max_epochs,
#                            verbose=1,  # Adjust verbosity as needed
#                            callbacks=[early_stopping])

#             # Test set predictions for this model
#             test_predictions = main_model.predict([X1_test, X2_test]).reshape(-1)
#             ensemble_predictions.append(test_predictions)

#         # Average the predictions from the ensemble
#         avg_test_predictions = np.mean(ensemble_predictions, axis=0)

#         # Store the average predictions and calculate performance metrics
#         test_result = pd.DataFrame({'y_pred': avg_test_predictions, 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

#         test_spearman = spearmanr(test_result.y_true, test_result.y_pred)[0]
#         daily_test_spearman = test_result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])
#         test_r2 = r2_score(test_result.y_true, test_result.y_pred)
#         test_mse = mean_squared_error(test_result.y_true, test_result.y_pred)

#         # Store test set performance
#         print(f'{time.time()-start:.2f} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | '
#               f'Test SP-coef: {test_spearman:6.2%} | Test R^2:{test_r2:6.2%} | Test MSE {test_mse:6.2%}')

#         scores.append([l1_reg, units, n_factors, fold, 'N/A', 'N/A', 'N/A', 'N/A', test_r2, test_mse, 'Test'])

#     # Convert scores to a DataFrame and append to the new HDF5 file
#     scores_df = pd.DataFrame(scores, columns=['l1_reg', 'units', 'n_factors', 'fold', 'spearman', 'daily_spearman_mean', 'daily_spearman_std', 'daily_spearman_median', 'r2_score', 'mse', 'set'])
#     scores_df.to_hdf(new_hdf5_path, key=f'{l1_reg}/{n_factors}', append=True, format='table')

# print("Training and test predictions complete.")


In [ ]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

scores = []
with pd.HDFStore(new_results_path / 'new_scores.h5') as store:
    for key in store.keys():
        scores.append(store[key])
scores = pd.concat(scores)

In [ ]:
scores.info()

In [ ]:
# Sort by 'n_factors' and 'r2_score' in descending order
sorted_avg = avg.sort_values(by=['n_factors', 'r2_score'], ascending=[True, False])

# Drop duplicates, keeping the first entry of each 'n_factors' group
highest_r2_per_factor = sorted_avg.drop_duplicates(subset=['n_factors'], keep='first')

# Display the resulting DataFrame
print(highest_r2_per_factor)


In [ ]:
# import pandas as pd

# # Sample data as per the provided example
# data = {
#     'n_factors': [1, 2, 3, 4, 5, 6],
#     'l1_reg': [1.000000e-07, 1.000000e-08, 1.000000e-05, 1.000000e-08, 1.000000e-05, 1.000000e-05],
#     'r2_score': [0.236430, 0.238089, 0.246361, 0.252998, 0.253971, 0.258462]
# }

# # Create DataFrame
# highest_r2_per_factor = pd.DataFrame(data)

# Extract the required metrics
r2_scores = highest_r2_per_factor['r2_score'].tolist()
l1_regs = highest_r2_per_factor['l1_reg'].tolist()

# Create LaTeX table with placeholders to avoid conflicting with LaTeX's use of braces
latex_table_template = """
\\begin{{table}}[ht]
\\centering
\\caption{{Model Performance Metrics for CA\\_0}}
\\label{{tab:metrics_ca_0}}
\\begin{{tabular}}{{lrrrrrr}}
\\toprule
 & \\multicolumn{{6}}{{c}}{{CA\\_0}} \\\\
\\cmidrule(r){{2-7}}
Metric & 1 & 2 & 3 & 4 & 5 & 6 \\\\
\\midrule
R^2 & {r2_1:.6f} & {r2_2:.6f} & {r2_3:.6f} & {r2_4:.6f} & {r2_5:.6f} & {r2_6:.6f} \\\\
l1\\_reg & {l1_1:.6e} & {l1_2:.6e} & {l1_3:.6e} & {l1_4:.6e} & {l1_5:.6e} & {l1_6:.6e} \\\\
\\bottomrule
\\end{{tabular}}
\\end{{table}}
"""

# Use format to insert the values into the placeholders
latex_table = latex_table_template.format(
    r2_1=r2_scores[0], r2_2=r2_scores[1], r2_3=r2_scores[2],
    r2_4=r2_scores[3], r2_5=r2_scores[4], r2_6=r2_scores[5],
    l1_1=l1_regs[0], l1_2=l1_regs[1], l1_3=l1_regs[2],
    l1_4=l1_regs[3], l1_5=l1_regs[4], l1_6=l1_regs[5]
)

# Print the LaTeX table
print(latex_table)


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
from scipy.optimize import minimize
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import spearmanr
# Setup the training configuration
max_epochs = 100
units = 32
batch_size = 32
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)


# Extract unique combinations of l1_reg and n_factors
selected_combinations = highest_r2_per_factor[['l1_reg', 'n_factors']].drop_duplicates().to_records(index=False)


YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=2*YEAR, val_period_length=1*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

start = time.time()

# Create the HDF5 file initially in 'table' format if it doesn't exist
with pd.HDFStore(str(results_path / 'scores.h5'), mode='w', complevel=9, complib='blosc:zstd') as store:
    store.close()

# Iterate over the selected combinations only
for l1_reg, n_factors in selected_combinations:
    scores = []
    main_model, factor_network_model = make_model(hidden_units=units, n_factors=n_factors, return_components=True, l1_reg=l1_reg)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

        # Train the main model
        main_model.fit([X1_train, X2_train], y_train,
                       batch_size=batch_size,
                       validation_data=([X1_val, X2_val], y_val),
                       epochs=max_epochs,
                       verbose=1,  # Adjust verbosity as needed
                       callbacks=[early_stopping])

        # Test set predictions and performance metrics
        test_predictions = main_model.predict([X1_test, X2_test]).reshape(-1)
        test_result = pd.DataFrame({'y_pred': test_predictions, 'y_true': y_test.stack().values}, index=y_test.stack().index).replace(-2, np.nan).dropna()

        test_spearman = spearmanr(test_result.y_true, test_result.y_pred)[0]
        daily_test_spearman = test_result.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])
        test_r2 = r2_score(test_result.y_true, test_result.y_pred)
        test_mse = mean_squared_error(test_result.y_true, test_result.y_pred)

        # Store test set performance
        print(f'{time.time()-start:.2f} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | '
              f'Test SP-coef: {test_spearman:6.2%} | Test R^2:{test_r2:6.2%} | Test MSE {test_mse:6.2%}')

        scores.append([l1_reg, units, n_factors, fold, 'N/A', 'N/A', 'N/A', 'N/A', test_r2, test_mse, 'Test'])

    scores_df = pd.DataFrame(scores, columns=['l1_reg', 'units', 'n_factors', 'fold', 'spearman', 'daily_spearman_mean', 'daily_spearman_std', 'daily_spearman_median', 'r2_score', 'mse', 'set'])
    scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{l1_reg}/{n_factors}', append=True, format='table')

print("Training and test predictions complete.")


**showing the largest MSE for each n_factor and l1_reg combination**

In [ ]:
# First, sort the 'avg' DataFrame by 'n_factors' and 'mse' in descending order
avg_sorted = avg.sort_values(by=['n_factors', 'ic_daily_median'], ascending=[True, False])

# Then, drop duplicates to keep the first (highest MSE) entry for each 'n_factors' and 'l1_reg' combination
top_mse_per_factor_l1_reg = avg_sorted.drop_duplicates(subset=['n_factors', 'l1_reg'])

# Now, top_mse_per_factor_l1_reg contains the top MSE for each combination of 'n_factors' and 'l1_reg'


In [ ]:
top_mse_per_factor_l1_reg

In [ ]:
top_combinations = []

# Iterate through each n_factors value
for factor in range(1, 7):
    # Filter 'avg_sorted' DataFrame for the current 'n_factors' and sort by 'r2_score' in descending order
    filtered_sorted = avg_sorted[avg_sorted['n_factors'] == factor].sort_values(by='ic_daily_median', ascending=False)

    # Pick the top row (highest 'r2_score')
    top_row = filtered_sorted.head(1)

    # Append the top row to 'top_combinations'
    top_combinations.append(top_row)

# Concatenate all top rows into a single DataFrame
top_combinations_df = pd.concat(top_combinations, ignore_index=True)

# Now, 'top_combinations_df' contains the top 'r2_score' and 'l1_reg' combination for each 'n_factors'


In [ ]:
print(top_combinations_df)

In [ ]:
# top = (avg.groupby(['n_factors'])
#        .apply(lambda x: x.nlargest(n=5, columns=['mse']))
#        .reset_index(-1, drop=True))

# top.nlargest(n=5, columns=['mse'])

**IC mean and median**

In [ ]:
fig, axes = plt.subplots(ncols=5, nrows=2, figsize=(20, 8), sharey='row', sharex=True)

# Loop through the range of n_factors (2 to 6)
for n in range(2, 7):
    # Filter the DataFrame for the current n_factor
    df = avg[avg.n_factors == n]

    # Plot 'ic_mean' against 'l1_reg' for the current n_factor
    df.plot(x='l1_reg', y='ic_mean', ax=axes[0][n-2], lw=1, title=f'{n} Factors', logx=True)
    axes[0][n-2].axhline(0, ls='--', c='k', lw=1)
    axes[0][n-2].get_legend().remove()
    axes[0][n-2].set_ylabel('IC Mean')
    axes[0][n-2].set_xlabel('L1 Regularization')

    # Plot 'ic_daily_median' against 'l1_reg' for the current n_factor
    df.plot(x='l1_reg', y='ic_daily_median', ax=axes[1][n-2], lw=1, logx=True)
    axes[1][n-2].axhline(0, ls='--', c='k', lw=1)
    axes[1][n-2].get_legend().remove()
    axes[1][n-2].set_ylabel('IC Daily Median')
    axes[1][n-2].set_xlabel('L1 Regularization')

# Adjust the layout, titles, and save the figure as before
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', title='# Units')
fig.suptitle('Cross-Validation Performance (2015-2019)', fontsize=16)
fig.tight_layout()
fig.subplots_adjust(top=.9)
fig.savefig(results_path / 'cv_performance', dpi=300)


**MSE and R^2**

In [ ]:
fig, axes = plt.subplots(ncols=5, nrows=2, figsize=(20, 8), sharey='row', sharex=True)

# Loop through the range of n_factors (2 to 6)
for n in range(2, 7):
    # Filter the DataFrame for the current n_factor
    df = avg[avg.n_factors == n]

    # Plot 'ic_mean' against 'l1_reg' for the current n_factor
    df.plot(x='l1_reg', y='mse', ax=axes[0][n-2], lw=1, title=f'{n} Factors', logx=True)
    axes[0][n-2].axhline(0, ls='--', c='k', lw=1)
    axes[0][n-2].get_legend().remove()
    axes[0][n-2].set_ylabel('MSE')
    axes[0][n-2].set_xlabel('L1 Regularization')

    # Plot 'ic_daily_median' against 'l1_reg' for the current n_factor
    df.plot(x='l1_reg', y='ic_daily_median', ax=axes[1][n-2], lw=1, logx=True)
    axes[1][n-2].axhline(0, ls='--', c='k', lw=1)
    axes[1][n-2].get_legend().remove()
    axes[1][n-2].set_ylabel('R^2 score')
    axes[1][n-2].set_xlabel('L1 Regularization')

# Adjust the layout, titles, and save the figure as before
handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', title='# Units')
fig.suptitle('Cross-Validation Performance (2015-2019)', fontsize=16)
fig.tight_layout()
fig.subplots_adjust(top=.9)
fig.savefig(results_path / 'cv_performance', dpi=300)


**predictions with l1_reg = 0.0001**

In [ ]:
import time
from tensorflow.keras.callbacks import Callback
from scipy.optimize import minimize
import numpy as np
early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

cols = ['l1_reg','units', 'n_factors', 'fold', 'ic_mean','ic_daily_mean', 'ic_daily_std', 'ic_daily_median', 'r2_score', 'mse']



max_epochs = 100
factor_opts = [1, 2, 3, 4, 5, 6]

l1_reg = [0.0001]

param_grid = list(product(l1_reg, factor_opts))
units=32
batch_size = 32

YEAR = 12
cv_merged = MultipleTimeSeriesCV(n_splits=8, train_period_length=5*YEAR,val_period_length=3*YEAR, test_period_length=1*YEAR, lookahead=1*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=8, train_period_length=5*YEAR,val_period_length=3*YEAR , test_period_length=1*YEAR, lookahead=1*YEAR)

# define accumulated results dataframe to evaluate model predictions for each model specification
accumulated_results = pd.DataFrame()
start = time.time()

for l1_reg, n_factors in param_grid:
    scores = []
    accumulated_scores = []
    main_model, factor_network_model = make_model(hidden_units=32, n_factors=n_factors, return_components=True, l1_reg=l1_reg)

    for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(loaded_portfolio_rets))):
        X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, loaded_portfolio_rets, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)
        ##Training the whole model
        # In your training loop
        main_model.fit([X1_train, X2_train], y_train,
                  batch_size=batch_size,
                  validation_data=([X1_val, X2_val], y_val),
                  epochs=max_epochs,
                  verbose=1,  # Adjust verbosity as needed
                  callbacks=[early_stopping])  # Include both callbacks

        result = (pd.DataFrame({'y_pred': main_model.predict([X1_test,
                                                        X2_test]).reshape(-1),
                                'y_true': y_test.stack().values},
                              index=y_test.stack().index)
                  .replace(-2, np.nan).dropna())

        # Assign the current 'n_factors' and 'l1_reg' to the result DataFrame
        result['n_factors'] = n_factors
        result['l1_reg'] = l1_reg


        accumulated_results = pd.concat([accumulated_results, result], ignore_index=False)

      #   #Calculate spearmans coefficient for val set
      #   r0 = spearmanr(accumulated_results.y_true, accumulated_results.y_pred)[0]
      #   # Calculate the daily spearman's coefficient
      #   r1 = accumulated_results.groupby(level='date').apply(lambda x: spearmanr(x.y_pred,x.y_true)[0])
      #   # calculate R^2
      #   r2 = r2_score(accumulated_results.y_true, accumulated_results.y_pred)
      #  # Calculate MSE
      #   mse = mean_squared_error(accumulated_results.y_true, accumulated_results.y_pred)

      #   print(f'{format_time(time.time()-start)} | factor:{n_factors} |lasso:{l1_reg}| fold:{fold:02} | sp-coe: {r0:6.2%} | '
      #         f'Avg sp {r1.mean():6.2%} | Median-Sp {r1.median():6.2%}| R^2:{r2:6.2%}| MSE {mse:6.2%}' )

# accumulated scores
##########################################################################
    # Perform calculations after accumulating results from all folds
    r0 = spearmanr(accumulated_results.y_true, accumulated_results.y_pred)[0]
    r1 = accumulated_results.groupby(level='date').apply(lambda x: spearmanr(x.y_pred, x.y_true)[0])
    r2 = r2_score(accumulated_results.y_true, accumulated_results.y_pred)
    mse = mean_squared_error(accumulated_results.y_true, accumulated_results.y_pred)

    # Print the metrics after all folds for the current combination of n_factors and l1_reg
    print(f'{format_time(time.time() - start)} | factor:{n_factors} | lasso:{l1_reg} | '
          f'Sp-coef: {r0:6.2%} | Avg Sp: {r1.mean():6.2%} | Median-Sp: {r1.median():6.2%} | '
          f'R^2: {r2:6.2%} | MSE: {mse:6.2%}')

    accumulated_scores.append([l1_reg ,units, n_factors, fold, r0,
                      r1.mean(), r1.std(), r1.median(), r2, mse])


    accumulated_scores = pd.DataFrame(accumulated_scores, columns=cols)
    accumulated_scores.to_hdf(results_path / 'accumulated_scores.h5', f'{l1_reg}/{n_factors}')


    # Assuming 'results_path' is a defined variable that specifies the directory where you want to save the file
    results_file_path = results_path / 'accumulated_results.h5'  # Define the file name and path

    # Save 'accumulated_results' DataFrame to an HDF5 file
    accumulated_results.to_hdf(results_file_path, key='accumulated_results', mode='w')





**loading predicted values for all factors**

In [ ]:
# Assuming 'results_file_path' is the path to the saved HDF5 file
accumulated_results = pd.read_hdf(results_file_path, key='accumulated_results')


**loading scores for prediction evaluation**

In [ ]:
import pandas as pd

accumulated_scores = []
with pd.HDFStore(results_path / 'accumulated_scores.h5') as store:
    for key in store.keys():
        accumulated_scores.append(store[key])
accumulated_scores = pd.concat(accumulated_scores)

In [ ]:
print(accumulated_scores)

In [ ]:
print(accumulated_results)

In [ ]:
# Find the minimum date
min_date = accumulated_results.index.get_level_values('date').min()
print(f"Minimum Date: {min_date}")

# Find the maximum date
max_date = accumulated_results.index.get_level_values('date').max()
print(f"Maximum Date: {max_date}")


In [ ]:
def get_train_valid_data_combined(merged_data, portfolio_data, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio):
    # Process merged_df_padded data
    train_merged = merged_data.iloc[train_idx_merged]
    val_merged = merged_data.iloc[val_idx_merged]
    test_merged = merged_data.iloc[test_idx_merged]

    X1_train = train_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)
    X1_val = val_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)
    X1_test = test_merged.loc[:, narratives].values.reshape(-1, n_tickers, n_narratives)

    y_train = train_merged.ret_excess.unstack('permno')
    y_val = val_merged.ret_excess.unstack('permno')
    y_test = test_merged.ret_excess.unstack('permno')

    # Get date ranges for merged data
    train_dates_merged = train_merged.index.get_level_values('date').unique()
    val_dates_merged = val_merged.index.get_level_values('date').unique()
    test_dates_merged = test_merged.index.get_level_values('date').unique()

    print(f"Train dates merged: {train_dates_merged.min()} to {train_dates_merged.max()}")
    print(f"Validation dates merged: {val_dates_merged.min()} to {val_dates_merged.max()}")
    print(f"Test dates merged: {test_dates_merged.min()} to {test_dates_merged.max()}")

    # Process loaded_portfolio_rets data
    X2_train = portfolio_data.loc[train_dates_portfolio]
    X2_val = portfolio_data.loc[val_dates_portfolio]
    X2_test = portfolio_data.loc[test_dates_portfolio]

    # Get date ranges for portfolio data
    print(f"Train dates portfolio: {train_dates_portfolio.min()} to {train_dates_portfolio.max()}")
    print(f"Validation dates portfolio: {val_dates_portfolio.min()} to {val_dates_portfolio.max()}")
    print(f"Test dates portfolio: {test_dates_portfolio.min()} to {test_dates_portfolio.max()}")

    return X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test

# Define cross-validation configurations
# YEAR = 12
# cv_merged = MultipleTimeSeriesCV(n_splits=10, train_period_length=20*YEAR, val_period_length=10*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)
# cv_portfolio = DateTimeSeriesCV(n_splits=10, train_period_length=20*YEAR, val_period_length=10*YEAR, test_period_length=1*YEAR, lookahead=0*YEAR)

# for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in tqdm(enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))), total=cv_merged.get_n_splits(merged_df_padded), desc="Processing folds", leave=True, ncols=100):
#     X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

from tqdm import tqdm

# Define cross-validation configurations
YEAR = 12
n_splits = 5
cv_merged = MultipleTimeSeriesCV(n_splits=n_splits, train_period_length=28*YEAR, val_period_length=2*YEAR, test_period_length=2*YEAR, lookahead=0*YEAR)
cv_portfolio = DateTimeSeriesCV(n_splits=n_splits, train_period_length=28*YEAR, val_period_length=2*YEAR, test_period_length=2*YEAR, lookahead=0*YEAR)

# Using tqdm to track the progress of the loop
for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in tqdm(enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))), total=n_splits, desc="Processing folds", leave=True, ncols=100):
    X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)

# for fold, ((train_idx_merged, val_idx_merged, test_idx_merged), (train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)) in enumerate(zip(cv_merged.split(merged_df_padded), cv_portfolio.split(x_t_all_months))):
#     X1_train, X2_train, y_train, X1_val, X2_val, y_val, X1_test, X2_test, y_test = get_train_valid_data_combined(merged_df_padded, x_t_all_months, train_idx_merged, val_idx_merged, test_idx_merged, train_dates_portfolio, val_dates_portfolio, test_dates_portfolio)
